In [ ]:
###
# ライブラリの準備
###

!pip -q install ultralytics roboflow


In [ ]:
###
# セットアップ
###

from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")
except Exception:
    ROOT_PATH = Path.cwd()

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

print("ROOT_PATH:", ROOT_PATH)


In [ ]:
###
# Cell 01
# Roboflowのデータセット取得コードを貼り付ける
###

## ここにコピペ


In [ ]:
###
# Cell 02
# YOLO用のdata.yamlを確認する
###

import yaml

DATA_YAML = Path(dataset.location) / "data.yaml"

# 今回は全画像をtrainに置くため、val/testもtrainを参照させる
def split_or_train(value):
    if not value:
        return "train/images"
    split_path = Path(value)
    if not split_path.is_absolute():
        split_path = Path(dataset.location) / split_path
    return value if split_path.exists() else "train/images"

data = yaml.safe_load(DATA_YAML.read_text())
data["path"] = str(Path(dataset.location))
data["train"] = "train/images"
data["val"] = split_or_train(data.get("val"))
data["test"] = split_or_train(data.get("test"))
DATA_YAML.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))

print("dataset:", dataset.location)
print("data.yaml:", DATA_YAML)
print(DATA_YAML.read_text())


In [ ]:
###
# Cell 03
# yolov8n.ptを初期値にしてface検出モデルを学習する
###

import torch
from ultralytics import YOLO

print("CUDA available:", torch.cuda.is_available())

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=512,
    batch=16,
    patience=10,
    project="runs/face",
    name="yolov8n_transfer",
    exist_ok=True,
)


In [ ]:
###
# Cell 04
# 学習結果を確認し、最良モデルを読み込む
###

from IPython.display import Image, display

run_dir = Path(results.save_dir)
best_model_path = run_dir / "weights" / "best.pt"

print("run_dir:", run_dir)
print("best model:", best_model_path)

display(Image(filename=str(run_dir / "results.png")))

best_model = YOLO(str(best_model_path))


In [ ]:
###
# Cell 05
# 学習画像で推論する
###

# 本当は未知の画像で確認する方がよい
from PIL import Image as PILImage

image_dir = Path(dataset.location) / "train" / "images"
sample_images = sorted(image_dir.glob("*"))[:5]

predict_results = best_model.predict(
    source=[str(path) for path in sample_images],
    conf=0.25,
)

for result in predict_results:
    display(PILImage.fromarray(result.plot()[..., ::-1]))


In [ ]:
###
# Cell 06
# Webカメラで画像を撮影する
###

from utils.camera import take_photo

photo_path = take_photo("face_camera.jpg")
display(Image(filename=photo_path))


In [ ]:
###
# Cell 07
# 撮影した画像でface検出を試す
###

from PIL import Image as PILImage

camera_results = best_model.predict(
    source="face_camera.jpg",
    conf=0.25,
)

display(PILImage.fromarray(camera_results[0].plot()[..., ::-1]))
